<a href="https://colab.research.google.com/github/cerr/pycerr-notebooks/blob/main/03_autosegmentation/generate_consensus_contour.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Consensus contours from multiple observers

This tutorial demonstrates how to combine several segmentations of the *same* structure - drawn by different observers or produced by different auto-segmentation models - into a single **consensus** contour using pyCERR's `cerr.contour.structure_consensus` module.

The module is the pyCERR counterpart of MATLAB CERR's `structCompare` and computes, for structures that all delineate the same target on the same scan:

- a **STAPLE** probabilistic consensus (Warfield et al., IEEE TMI 2004) with each observer's estimated sensitivity and specificity,
- the per-voxel **observer-agreement fraction**,
- **Fleiss' kappa** inter-observer agreement, and
- agreement / volume statistics at a range of agreement thresholds.

It can then write a consensus segmentation back into the plan container using any of these methods: `staple`, `majority`, `agreement`, `union`, `intersection`.

## I/O

- **Input:** a scan (CT/MR/...) and two or more structures on that scan, one per observer.
- **Output:** one or more consensus structures added to the plan container (viewable and exportable like any other structure).

> This notebook uses **placeholder data paths** and contains **no PHI**. A self-contained synthetic demo (built on the anonymized phantom CT that ships with pyCERR) lets you run every cell without supplying any data.

## License

By downloading the software you are agreeing to the following terms and conditions as well as to the Terms of Use of CERR software.

**`THE SOFTWARE IS PROVIDED "AS IS" AND CERR DEVELOPMENT TEAM AND ITS COLLABORATORS DO NOT MAKE ANY WARRANTY, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE, NOR DO THEY ASSUME ANY LIABILITY OR RESPONSIBILITY FOR THE USE OF THIS SOFTWARE.`**

`This software is for research purposes only and has not been approved for clinical use.`

`Software has not been reviewed or approved by the Food and Drug Administration, and is for non-clinical, IRB-approved Research Use Only. In no event shall data or images generated through the use of the Software be used in the provision of patient care.`

## Install pyCERR

In [ ]:
%%capture
!pip install "pyCERR @ git+https://github.com/cerr/pyCERR.git@main"

## Data paths

Point `dataDir` at a directory that contains your scan (e.g. DICOM CT) **and** two or more structures of the same target, one per observer (for example, one RTSTRUCT per observer, or a multi-ROI RTSTRUCT).

Set `USE_SYNTHETIC = False` to use your own data; leave it `True` to run the built-in synthetic demo (no data required).

In [ ]:
# ---- EDIT THESE ----
USE_SYNTHETIC = True                       # False -> use your own data
dataDir       = r'/path/to/observer_dicom' # scan + >=2 observer structs

# Names of the observer structures to combine (as they appear in the data).
# Leave as None to use every structure on the scan.
observerNames = None                       # e.g. ['obs_A', 'obs_B', 'obs_C']

## Load the scan and observer structures

The synthetic branch loads the anonymized phantom CT bundled with pyCERR and adds four synthetic "observer" contours of the same target (three concordant, one deliberate outlier) so the consensus statistics are illustrative. The real-data branch simply loads `dataDir`.

In [ ]:
import os
import numpy as np
import cerr
from cerr import plan_container as pc
import cerr.contour.rasterseg as rs
from cerr.dataclasses import structure as structr

if USE_SYNTHETIC:
    # Anonymized IBSI-style phantom CT that ships with pyCERR (no PHI).
    phantomDir = os.path.join(os.path.dirname(cerr.__file__), 'datasets',
                              'radiomics_phantom_dicom', 'pat_1')
    planC = pc.loadDcmDir(phantomDir)

    nr, ncol, ns = [int(x) for x in planC.scan[0].getScanSize()]
    cr, cc, cs = nr // 2, ncol // 2, ns // 2   # center (row, col, slice)
    rows, cols, slcs = np.indices((nr, ncol, ns))

    def _blob(dr, dc, rIn, rSl):
        # solid ellipsoid: in-plane radius rIn voxels, through-slice rSl
        return (((rows - (cr + dr)) ** 2 + (cols - (cc + dc)) ** 2)
                / float(rIn) ** 2
                + ((slcs - cs) / float(rSl)) ** 2) <= 1.0

    # three concordant observers + one shifted/smaller outlier
    obs = {
        'observer_A': _blob(0,   0,  22, 5),
        'observer_B': _blob(2,   0,  22, 5),
        'observer_C': _blob(0,  -2,  20, 5),
        'observer_D': _blob(12, 12,  14, 4),  # outlier
    }
    for name, mask in obs.items():
        planC = structr.importStructureMask(mask, 0, name, planC, None)
    observerNames = list(obs.keys())
else:
    planC = pc.loadDcmDir(dataDir)

allNames = [s.structureName for s in planC.structure]
print('Structures in planC:', allNames)

## Select the observer structures to combine

In [ ]:
import cerr.dataclasses.scan as scn

if observerNames:
    structNumV = [allNames.index(n) for n in observerNames]
else:
    # every structure associated with scan 0
    structNumV = [i for i, s in enumerate(planC.structure)
                  if scn.getScanNumFromUID(s.assocScanUID, planC) == 0]

print('Combining structures:', structNumV,
      '->', [allNames[i] for i in structNumV])
assert len(structNumV) >= 2, 'Need at least two observer structures.'

## Compare the observers (STAPLE, kappa, agreement statistics)

`compareStructures` returns a dictionary of results; `summaryText` renders it as a readable report. Note how the outlier observer (D) gets a low STAPLE sensitivity.

In [ ]:
from cerr.contour import structure_consensus as consensus

result = consensus.compareStructures(structNumV, planC)
print(consensus.summaryText(result))

You can also access the raw fields, e.g. the per-voxel STAPLE probability map, the observer-agreement fraction, and per-observer sensitivity/specificity:

In [ ]:
print('STAPLE probability map shape :', result['staple3M'].shape)
print('Max agreement fraction       :', float(result['agreementFraction3M'].max()))
print('Fleiss kappa                 : %.4f' % result['kappa'])
for name, se, sp in zip(result['structNames'],
                        result['sensitivity'], result['specificity']):
    print('  %-12s sensitivity=%.3f  specificity=%.3f' % (name, se, sp))

## Generate consensus contours with various methods

| Method | Definition |
|--------|------------|
| `staple` | STAPLE probability >= `threshold` |
| `majority` | included by more than half of the observers |
| `agreement` | observer-agreement fraction >= `threshold` |
| `union` | included by at least one observer |
| `intersection` | included by all observers |

Each call adds a new structure to `planC`. We reuse the `result` computed above so STAPLE is not recomputed each time.

In [ ]:
methods = [
    ('staple',       0.5),
    ('majority',     None),
    ('agreement',    0.5),   # >= 50% of observers
    ('union',        None),
    ('intersection', None),
]

consensusStructNums = {}
for method, thr in methods:
    kwargs = {} if thr is None else {'threshold': thr}
    planC, newNum = consensus.createConsensusStructure(
        structNumV, planC, method=method, result=result, **kwargs)
    consensusStructNums[method] = newNum
    vol_cc = rs.getStrMask(newNum, planC).sum() * result['voxelVolume_cc']
    print('%-13s -> struct %2d  %-24s  %8.2f cc'
          % (method, newNum, planC.structure[newNum].structureName, vol_cc))

## Visualize

First the individual observer contours (note the outlier), then the consensus contours. `showNB` returns a live viewer - scroll through slices with the slider.

In [ ]:
import numpy as np
from cerr.viewer.pycerr_nbviewer import showNB

# Observer contours
showNB(planC=planC, scanNum=0, structNums=structNumV,
       windowCenter=40, windowWidth=400)

In [ ]:
# Consensus contours (majority, STAPLE, intersection)
showStructs = [consensusStructNums['majority'],
               consensusStructNums['staple'],
               consensusStructNums['intersection']]
showNB(planC=planC, scanNum=0, structNums=showStructs,
       windowCenter=40, windowWidth=400)

## Next steps

- Tune the STAPLE / agreement `threshold` to make the consensus tighter or looser.
- Export a consensus structure to DICOM RTSTRUCT with `cerr.dcm_export`, or to NIfTI with the structure's `saveNii`.
- The same tool is available interactively in the pyCERR GUI under **Tools > Structure consensus (compare/STAPLE)...**.